# Statistics for Data Analysts/Engineers — Module 12: Linear Regression

In Module 11 we learned to measure HOW STRONG the relationship between two variables
is (correlation). **Linear regression** goes one step further: it builds a concrete
mathematical model that lets you PREDICT the value of one variable from another (or
several others) -- with a concrete equation, concrete coefficients, and a measure of
how well the model fits the data.

## Table of contents
1. [Simple linear regression](#sec1)
2. [R-squared: a measure of fit](#sec2)
3. [Residual diagnostics](#sec3)
4. [Multiple linear regression](#sec4)
5. [Adjusted R²](#sec5)
6. [Predicting new values](#sec6)
7. [Summary and exercises](#sec7)


Every module in this course starts by generating the same synthetic dataset -- thanks to a fixed random seed (`np.random.default_rng(42)`), the numbers always come out identical, so you can open any module independently of the others, even after restarting the kernel.

In [ ]:
import numpy as np
import pandas as pd

rng = np.random.default_rng(42)
n = 400

city = rng.choice(
    ["New York", "Chicago", "Austin", "Seattle", "Denver"],
    size=n, p=[0.30, 0.25, 0.15, 0.15, 0.15],
)
region = rng.choice(["North", "South", "East", "West"], size=n)
age = rng.integers(18, 70, size=n)

# monthly spend: right-skewed distribution (typical for spending data)
monthly_spend = rng.lognormal(mean=5.2, sigma=0.5, size=n).round(2)

# satisfaction correlated with spend + a region effect (deliberately different) + random
# noise, clipped to a 1-10 scale -- this gives Module 10 (ANOVA) a real between-group gap
region_effect = {"North": 0.0, "South": -0.5, "East": 0.3, "West": 0.8}
satisfaction = np.clip(
    monthly_spend / 45
    + np.array([region_effect[r] for r in region])
    + rng.normal(0, 1.5, size=n),
    1, 10,
).round(1)

# A/B campaign group -- with a REAL injected effect on conversion (group B better)
group = rng.choice(["A", "B"], size=n)
conversion_prob = np.where(group == "A", 0.10, 0.20)
converted = rng.binomial(1, conversion_prob)

# spend before/after a loyalty-program rollout (paired data)
spend_before = rng.normal(220, 40, size=n).round(2)
program_effect = rng.normal(15, 10, size=n)
spend_after = (spend_before + program_effect).round(2)

customers = pd.DataFrame({
    "customer_id": np.arange(1, n + 1),
    "city": city,
    "region": region,
    "age": age,
    "monthly_spend": monthly_spend,
    "satisfaction": satisfaction,
    "campaign_group": group,
    "converted": converted,
    "spend_before": spend_before,
    "spend_after": spend_after,
})
customers.head()


<a id="sec1"></a>
## 1. Simple linear regression

**Simple linear regression** looks for the best-fitting straight line `y = b0 + b1*x`,
describing how the dependent variable `y` (what we're predicting) depends on one
independent variable `x` (the predictor). "Best-fitting" means: minimizing the sum of
squared distances between the points and the line (*ordinary least squares*, OLS).

Let's check how `monthly_spend` (predictor) predicts `satisfaction` (dependent
variable) -- the exact same pair we studied via correlation in Module 11.

In [ ]:
from statsmodels.formula.api import ols

model = ols("satisfaction ~ monthly_spend", data=customers).fit()
print(model.params)


`Intercept` (`b0`) is the predicted value of `satisfaction` when `monthly_spend = 0`.
The coefficient on `monthly_spend` (`b1`, the *slope*) tells you how much the predicted
`satisfaction` changes when `monthly_spend` rises by one unit ($1). We get a full,
detailed model summary (standard errors, p-values, confidence intervals for every
coefficient) with `.summary()`:

In [ ]:
print(model.summary())


> 📊 **A coefficient's p-value is a significance test**
>
> The `monthly_spend` row in the table has its own p-value -- this is a test of H0: `b1 = 0` (the predictor has no linear effect on `y` at all). A very small p-value means the slope is significantly different from zero -- the exact same kind of hypothesis test from Module 7, just applied to a regression coefficient.

In [ ]:
import matplotlib.pyplot as plt
import numpy as np

plt.figure(figsize=(7, 5))
plt.scatter(customers["monthly_spend"], customers["satisfaction"], alpha=0.4, label="data")

x_line = np.linspace(customers["monthly_spend"].min(), customers["monthly_spend"].max(), 100)
y_line = model.params["Intercept"] + model.params["monthly_spend"] * x_line
plt.plot(x_line, y_line, color="red", linewidth=2, label="regression line")

plt.xlabel("Monthly spend")
plt.ylabel("Satisfaction")
plt.legend()
plt.show()


<a id="sec2"></a>
## 2. R-squared: a measure of fit

**R-squared** (`R^2`, the *coefficient of determination*) tells you what PERCENTAGE of
the variance in the dependent variable the model explains. `R^2 = 0` means the model
explains nothing (we might as well always predict the mean), `R^2 = 1` means a perfect
fit.

In [ ]:
print(f"R^2 = {model.rsquared:.4f}")


> 💡 **In simple regression, R² is just r² from Module 11**
>
> For a regression with ONE predictor, `R^2` is exactly the square of the Pearson correlation between `x` and `y` from Module 11 (`r^2`). That's no coincidence -- both measures rest on the same idea of "how well does a linear relationship describe the data", just viewed from two different angles.

In [ ]:
from scipy import stats

r, _ = stats.pearsonr(customers["monthly_spend"], customers["satisfaction"])
print(f"r^2 from Pearson correlation: {r ** 2:.4f}")
print(f"R^2 from regression: {model.rsquared:.4f}")


<a id="sec3"></a>
## 3. Residual diagnostics

**Residuals** are the differences between the actual values and the values predicted by
the model (`residual = y_actual - y_predicted`). A good linear regression assumes the
residuals are approximately normal and have constant variance regardless of the
predicted value (*homoscedasticity*) -- the absence of this is called
*heteroscedasticity*.

In [ ]:
residuals = model.resid
fitted = model.fittedvalues

plt.figure(figsize=(7, 5))
plt.scatter(fitted, residuals, alpha=0.4)
plt.axhline(0, color="red", linestyle="--")
plt.xlabel("Fitted values")
plt.ylabel("Residuals")
plt.title("Residuals vs. fitted values")
plt.show()


A cloud of points with no clear pattern (scattered randomly around the `0` line)
suggests the homoscedasticity assumption is fine. A funnel shape or a clear curve would
suggest a problem. Let's also check the residuals' normality with the Shapiro-Wilk test
(Module 8):

In [ ]:
sw_statistic, sw_p_value = stats.shapiro(residuals)
print(f"Shapiro-Wilk: statistic={sw_statistic:.4f}, p={sw_p_value:.4f}")


> ⚠️ **A formally non-significant normality test isn't always a problem**
>
> With `n=400`, the Shapiro-Wilk test's p-value can come out a bit below `0.05` (which is exactly what happens here) -- partly because `satisfaction` in our data was artificially clipped (`clip`) to the `[1, 10]` range, which slightly disturbs normality in the tails of the distribution. Statistical tests with a large sample detect even VERY small deviations from perfect normality (the exact same issue as with small effects in Module 7). In practice, you look at the residual plot (a histogram or a quantile-quantile plot) first, and only then at the formal test.

In [ ]:
import statsmodels.api as sm

sm.qqplot(residuals, line="45", fit=True)
plt.title("Quantile-quantile (Q-Q) plot of residuals")
plt.show()


<a id="sec4"></a>
## 4. Multiple linear regression

**Multiple linear regression** extends the same idea to MANY predictors at once:
`y = b0 + b1*x1 + b2*x2 + ...`. Let's add `region` (a categorical variable -- remember
`C(...)` from Module 10!) to the model from section 1:

In [ ]:
multi_model = ols("satisfaction ~ monthly_spend + C(region)", data=customers).fit()
print(multi_model.params)
print(f"R^2 = {multi_model.rsquared:.4f}")


`R^2` went up compared to the single-predictor model (section 2) -- region really does
carry additional information, exactly as the box plot from Module 2 and the ANOVA from
Module 10 suggested. The coefficients for individual regions (`C(region)[T.South]` etc.)
show the difference relative to the baseline category (the one without its own row --
`statsmodels` picks it automatically in alphabetical order) -- and broadly reflect the
same pattern of differences between regions that we built into the data ourselves back
in Module 8/10 (West highest, South lowest).

<a id="sec5"></a>
## 5. Adjusted R²

Plain `R^2` has a flaw: it grows (or at worst stays the same) EVERY time you add a new
predictor -- even a completely useless, random one. **Adjusted R²** penalizes adding
predictors that don't really improve the model, so it's better suited for comparing
models with different numbers of variables.

In [ ]:
print(f"R^2 (spend + region): {multi_model.rsquared:.4f}")
print(f"Adjusted R^2 (spend + region): {multi_model.rsquared_adj:.4f}")


> 📊 **If adjusted R² drops, the predictor probably isn't helping**
>
> Adding a predictor that truly adds nothing (e.g. pure noise) usually makes plain `R^2` grow by a microscopic fraction, while adjusted `R^2` actually DROPS -- because the penalty for the extra parameter outweighs the negligible gain in fit. You'll see this yourself in exercise 2 below.

<a id="sec6"></a>
## 6. Predicting new values

Once the model is fit, we can predict `satisfaction` for a brand-new customer we've
never seen -- just supply the predictor values in the same format as the training
data:

In [ ]:
new_customer = pd.DataFrame({"monthly_spend": [200.0], "region": ["West"]})
predicted_satisfaction = multi_model.predict(new_customer)
print(f"Predicted satisfaction: {predicted_satisfaction.values[0]:.2f}")


> ⚠️ **Extrapolating beyond the training data's range is risky**
>
> The model only "knows" the range of values it was trained on. Predicting for `monthly_spend = 100000` (far outside our data's range) could give an absurd result -- a straight line, fit locally, doesn't have to match the data's behavior far beyond the observed range at all.

<a id="sec7"></a>
## 7. Summary and exercises

In this module we covered:
- simple linear regression (`statsmodels.formula.api.ols`) and interpreting its
  coefficients (`Intercept`, slope),
- `R^2` as a measure of fit, and its relationship to the Pearson correlation from
  Module 11,
- residual diagnostics: residuals-vs-fitted plot (homoscedasticity), the Shapiro-Wilk
  test, and a Q-Q plot (normality),
- multiple regression with categorical predictors (`C(...)`),
- adjusted `R^2` as a better measure for comparing models with different numbers of
  predictors,
- predicting for new observations with `.predict()`.

> 📝 **Exercise 1: Regressing post-program spend on pre-program spend**
>
> Build a simple regression `spend_after ~ spend_before` (the data from Module 8). Print the coefficients and `R^2`. Is the slope close to `1`? What does that suggest about the relationship between these two variables?

<details>
<summary><b>👉 Click to reveal a sample solution</b></summary>

```python
before_after_model = ols("spend_after ~ spend_before", data=customers).fit()
print(before_after_model.params)
print(f"R^2 = {before_after_model.rsquared:.4f}")
```

The slope comes out close to 1, and R^2 is very high (around 0.95) -- this is expected, since in Module 8 we built spend_after as spend_before plus a random effect with a mean of about 15. The intercept is in that same ballpark (though not identical -- it's just a fit to one particular sample), which confirms that construction.
</details>

> 📝 **Exercise 2: Does age improve the model?**
>
> Build a model `satisfaction ~ monthly_spend + age` and compare its `R^2` and adjusted `R^2` with the model from section 1 (`monthly_spend` alone). Also check the p-value of the `age` coefficient. Does `age` genuinely improve the model?

<details>
<summary><b>👉 Click to reveal a sample solution</b></summary>

```python
model_with_age = ols("satisfaction ~ monthly_spend + age", data=customers).fit()
print(f"R^2 (without age): {model.rsquared:.4f}, adjusted: {model.rsquared_adj:.4f}")
print(f"R^2 (with age): {model_with_age.rsquared:.4f}, adjusted: {model_with_age.rsquared_adj:.4f}")
print(f"p-value for age: {model_with_age.pvalues['age']:.4f}")
```

Plain R^2 usually grows by a microscopic fraction, but adjusted R^2 actually DROPS slightly, and age's p-value is well above 0.05 -- exactly the mechanism from section 5. Age is drawn in our data completely independently of everything else, so it adds no real predictive power.
</details>

> 📝 **Exercise 3: Residual diagnostics for the model with region**
>
> Draw a residuals-vs-fitted plot for `multi_model` (section 4, with region). Do you see any worrying pattern (e.g. a funnel shape)?

<details>
<summary><b>👉 Click to reveal a sample solution</b></summary>

```python
multi_residuals = multi_model.resid
multi_fitted = multi_model.fittedvalues

plt.figure(figsize=(7, 5))
plt.scatter(multi_fitted, multi_residuals, alpha=0.4)
plt.axhline(0, color="red", linestyle="--")
plt.xlabel("Fitted values")
plt.ylabel("Residuals")
plt.show()
```
</details>

> 📝 **Exercise 4: Predicting for several new customers at once**
>
> Use `multi_model.predict()` to predict `satisfaction` for THREE new customers at once (different combinations of `monthly_spend` and `region`), passing all of them at once in a single DataFrame.

<details>
<summary><b>👉 Click to reveal a sample solution</b></summary>

```python
new_customers = pd.DataFrame({
    "monthly_spend": [100.0, 250.0, 400.0],
    "region": ["North", "South", "West"],
})
multi_model.predict(new_customers)
```
</details>

> 🔥 **Challenge: does campaign group help once we control for spend?**
>
> Build the model `satisfaction ~ monthly_spend + C(region) + C(campaign_group)` -- the full model with three predictors. Check the p-value of the `campaign_group` coefficient. Compare the result to Module 10's two-way ANOVA challenge, which did NOT include spend -- did the conclusion about `campaign_group`'s significance change?

<details>
<summary><b>👉 Click to reveal a sample solution</b></summary>

```python
full_model = ols(
    "satisfaction ~ monthly_spend + C(region) + C(campaign_group)", data=customers
).fit()
print(f"p-value for campaign_group: {full_model.pvalues.filter(like='campaign_group').iloc[0]:.4f}")
print(f"R^2: {full_model.rsquared:.4f} (vs {multi_model.rsquared:.4f} without campaign group)")
```

Here (with spend added as a predictor) campaign_group does NOT come out significant (p clearly above 0.05) -- unlike the two-way ANOVA in Module 10, where (without spend in the model) it came out significant at p=0.04 as a false alarm (Type I error). This is a good illustration that whether a variable looks significant can depend on what OTHER variables are included in the model -- adding spend as a control predictor here 'absorbed' some of the noise that had previously landed, by chance, on campaign group.
</details>

## What's next?

Linear regression predicts a CONTINUOUS variable (e.g. satisfaction on a 1-10 scale).
But what if we want to predict something BINARY -- e.g. whether a customer converts
(yes/no)? In **Module 13** we'll meet **logistic regression**, designed exactly for
that purpose.